# EvalHub - Create Evaluation Job

This notebook demonstrates how to create an evaluation job using the EvalHub REST API.

## Prerequisites
- EvalHub deployed in `redhat-ods-applications` namespace
- Model deployed (e.g., `tinyllama-1b` in `models-as-a-service` namespace)
- Model API token/key available
- User has MLflow RBAC permissions (see RoleBinding in 13-mlflow)

In [ ]:
import requests
import json
import os
from urllib3.exceptions import InsecureRequestWarning
import warnings

# Suppress SSL warnings for demo purposes (use proper certs in production)
warnings.simplefilter('ignore', InsecureRequestWarning)

## Configuration

Set up the EvalHub endpoint and authentication

In [ ]:
# EvalHub API endpoint
EVALHUB_URL = "https://evalhub-redhat-ods-applications.apps.ocp.sandbox5548.opentlc.com"

# Get OpenShift token for authentication
# In a real environment, you would get this from:
# 1. Running: oc whoami -t
# 2. From the OpenShift web console (copy login command)
# 3. From environment variable: os.getenv('OPENSHIFT_TOKEN')
OCP_TOKEN = os.getenv('OPENSHIFT_TOKEN', 'sha256~YOUR_TOKEN_HERE')

# Model configuration
MODEL_NAMESPACE = "models-as-a-service"
MODEL_NAME = "tinyllama-1b"
MODEL_URL = f"https://maas.apps.ocp.sandbox5548.opentlc.com/{MODEL_NAMESPACE}/{MODEL_NAME}"

# Get model API token
# This should be the ServiceAccount token for the model
# You can get it with: kubectl get secret tinyllama-1b-sa-token -n models-as-a-service -o jsonpath='{.data.token}' | base64 -d
MODEL_API_TOKEN = os.getenv('MODEL_API_TOKEN', 'YOUR_MODEL_TOKEN_HERE')

# Request headers for EvalHub API
headers = {
    "Authorization": f"Bearer {OCP_TOKEN}",
    "Content-Type": "application/json"
}

print(f"✓ EvalHub URL: {EVALHUB_URL}")
print(f"✓ Model URL: {MODEL_URL}")
print(f"✓ Auth configured: {OCP_TOKEN[:20]}...")

## Step 1: Check EvalHub Health

In [ ]:
response = requests.get(
    f"{EVALHUB_URL}/api/v1/health",
    headers=headers,
    verify=False
)

print(f"Status Code: {response.status_code}")
print(f"Response: {response.text}")

## Step 2: List Available Providers

In [ ]:
response = requests.get(
    f"{EVALHUB_URL}/api/v1/evaluations/providers",
    headers=headers,
    verify=False
)

print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    providers = response.json()
    print(f"\nAvailable Providers:")
    for provider in providers.get('providers', []):
        print(f"  - {provider}")
else:
    print(f"Error: {response.text}")

## Step 3: List Available Collections

In [ ]:
response = requests.get(
    f"{EVALHUB_URL}/api/v1/evaluations/collections",
    headers=headers,
    verify=False
)

print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    collections = response.json()
    print(f"\nAvailable Collections:")
    for collection in collections.get('collections', []):
        print(f"  - {collection}")
else:
    print(f"Error: {response.text}")

## Step 4: Create Evaluation Job

Create an evaluation job for the Arc Easy benchmark

In [ ]:
# Evaluation job configuration
evaluation_job = {
    "name": "EvalHub Eval 01",
    "tenant": MODEL_NAMESPACE,
    "model": {
        "name": MODEL_NAME,
        "url": MODEL_URL,
        "api_key": MODEL_API_TOKEN  # Model authentication token
    },
    "evaluation": "Arc Easy",
    "provider": "lm-evaluation-harness",
    "limit": 10  # Limit to 10 samples for quick testing
}

print("Creating evaluation job with config:")
print(json.dumps({
    **evaluation_job,
    "model": {**evaluation_job["model"], "api_key": "<REDACTED>"}
}, indent=2))

In [ ]:
response = requests.post(
    f"{EVALHUB_URL}/api/v1/evaluations/jobs",
    headers=headers,
    json=evaluation_job,
    verify=False
)

print(f"Status Code: {response.status_code}")
print(f"\nResponse:")
print(json.dumps(response.json(), indent=2))

if response.status_code == 200 or response.status_code == 201:
    job_data = response.json()
    job_id = job_data.get('id', job_data.get('job_id'))
    print(f"\n✓ Evaluation job created successfully!")
    print(f"  Job ID: {job_id}")
    print(f"  Status: {job_data.get('status', 'unknown')}")
    
    # Save job ID for later use
    with open('evalhub_job_id.txt', 'w') as f:
        f.write(job_id)
    print(f"\n  Job ID saved to: evalhub_job_id.txt")
else:
    print(f"\n✗ Failed to create evaluation job")
    print(f"  Error: {response.text}")

## Troubleshooting

If you encounter errors:

1. **403 Forbidden**: Check that you have MLflow RBAC permissions:
   ```bash
   kubectl get rolebinding evalhub-mlflow-user-access -n models-as-a-service
   ```

2. **401 Unauthorized**: Verify your OpenShift token:
   ```bash
   oc whoami -t
   ```

3. **Model authentication issues**: Get the model token:
   ```bash
   kubectl get secret tinyllama-1b-sa-token -n models-as-a-service -o jsonpath='{.data.token}' | base64 -d
   ```

4. **Job stuck in Pending**: Check pod events:
   ```bash
   kubectl get pods -n models-as-a-service | grep eval
   kubectl describe pod <pod-name> -n models-as-a-service
   ```

5. **Known RHOAI 3.4.0 bug**: EvalHub may incorrectly use the API key token as a secret name instead of creating a proper secret reference. This causes jobs to fail with "secret not found" errors. See the GitOps repository documentation for workarounds.